# **Pipeline de Extracción de Entidades Técnicas**

Este notebook construye un pipeline con LangChain que recibe un texto libre
(ej. un log de error o descripción de arquitectura) y devuelve un objeto
validado con Pydantic: tecnologías mencionadas, nivel de criticidad y un
resumen técnico.

Proveedores soportados de forma intercambiable: **OpenAI, Anthropic, Gemini**.

In [ ]:
!pip install -q langchain langchain-core langchain-openai langchain-anthropic langchain-google-genai pydantic

##  **Configuración de API Keys**

Usamos Colab Secrets (ícono de llave 🔑 en el panel izquierdo) para no
exponer las keys en el notebook. Necesitás cargar `OPENAI_API_KEY`,
`ANTHROPIC_API_KEY` y `GOOGLE_API_KEY` como secrets antes de correr esta celda.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("🔑 OPENAI_API_KEY: ").strip()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("🔑 ANTHROPIC_API_KEY: ").strip()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("🔑 Ingresá tu GOOGLE_API_KEY (gratis en aistudio.google.com/apikey): ").strip()

🔑 OPENAI_API_KEY: ··········
🔑 ANTHROPIC_API_KEY: ··········
🔑 Ingresá tu GOOGLE_API_KEY (gratis en aistudio.google.com/apikey): ··········


## **Esquema Pydantic (el contrato de salida)**

Definimos la forma exacta que debe tener la respuesta del LLM.
`nivel_de_criticidad` es un Enum (no un string libre) para que el modelo
no pueda inventar valores como "high" o "ALTA". La lista de tecnologías
no puede quedar vacía ni con duplicados.

In [ ]:
from enum import Enum
from typing import List
from pydantic import BaseModel, Field, field_validator


class NivelCriticidad(str, Enum):
    BAJA = "baja"
    MEDIA = "media"
    ALTA = "alta"


class EntidadesTecnicas(BaseModel):
    tecnologias: List[str] = Field(
        ..., min_length=1,
        description="Lista de tecnologías, frameworks o herramientas mencionadas en el texto"
    )
    nivel_de_criticidad: NivelCriticidad = Field(
        ..., description="Gravedad del problema o relevancia de la arquitectura descripta"
    )
    resumen_tecnico: str = Field(
        ..., min_length=10,
        description="Resumen técnico de 1-2 oraciones sobre el contenido del texto"
    )

    @field_validator("tecnologias")
    @classmethod
    def sin_duplicados_ni_vacios(cls, v: List[str]) -> List[str]:
        limpio = [t.strip() for t in v if t.strip()]
        if not limpio:
            raise ValueError("La lista de tecnologías no puede quedar vacía tras limpiar")
        return list(dict.fromkeys(limpio))

## 3. Prompt modular con ChatPromptTemplate

Evitamos f-strings hardcodeados. `ChatPromptTemplate` declara `{texto}`
como variable de entrada, que LangChain resuelve al invocar la cadena.
Separamos instrucción del sistema (rol del analista) del mensaje humano
(el texto a procesar).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Sos un analista técnico. Extraé información estructurada del texto que te pasa el usuario. "
     "Identificá tecnologías mencionadas, evaluá el nivel de criticidad del problema o arquitectura "
     "descripta, y generá un resumen técnico breve."),
    ("human", "{texto}"),
])

## **Fábrica de modelos intercambiable**

Misma idea del Módulo 1: una función central decide qué proveedor
instanciar. `temperature=0` es intencional acá: para extracción
estructurada queremos determinismo, no creatividad.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI


def get_model(provider: str):
    if provider == "openai":
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)
    elif provider == "anthropic":
        return ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
    elif provider == "gemini":
        return ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)
    raise ValueError(f"Proveedor no soportado: {provider}")

## **Cadena LCEL con salida forzada y reintento**

`prompt | model.with_structured_output(Schema)` es la composición LCEL:
el prompt resuelto se pasa al modelo, y el modelo devuelve directamente
una instancia de `EntidadesTecnicas` (no texto crudo) gracias a
`with_structured_output`.

`.with_retry()` reintenta automáticamente si el LLM devuelve un JSON
mal formado o incompleto (por ejemplo, cortado por límite de tokens).

In [ ]:
def build_chain(provider: str = "openai"):
    model = get_model(provider)
    structured_model = model.with_structured_output(EntidadesTecnicas)

    chain = (prompt | structured_model).with_retry(
        stop_after_attempt=3,
        wait_exponential_jitter=True,
    )
    return chain

## **Función asíncrona con logs**

`process_text()` ejecuta la cadena con `.ainvoke()` (versión async,
no bloqueante) y deja logs del proceso, incluyendo errores si fallan
todos los reintentos.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("pipeline_extraccion")


async def process_text(text: str, provider: str = "openai") -> EntidadesTecnicas:
    chain = build_chain(provider)
    logger.info(f"[{provider}] Procesando texto ({len(text)} caracteres)...")

    try:
        resultado = await chain.ainvoke({"texto": text})
        logger.info(f"[{provider}] ✅ Extracción validada: {resultado.model_dump()}")
        return resultado
    except Exception as e:
        logger.error(f"[{provider}] ❌ Falló tras reintentos: {e}")
        raise

## **Prueba con los tres proveedores**

Corremos el mismo texto contra OpenAI, Anthropic y Gemini para comprobar
que el pipeline es intercambiable y que los tres devuelven un objeto
con la misma forma validada.

In [ ]:
texto_ejemplo = """
Nuestra API en FastAPI está devolviendo timeouts intermitentes. El caché en Redis
parece saturarse en picos de tráfico y las conexiones a PostgreSQL se agotan
porque el pool está mal dimensionado. Esto está afectando a usuarios en producción.
"""

for provider in ["openai", "anthropic", "gemini"]:
    try:
        resultado = await process_text(texto_ejemplo, provider=provider)
        print(f"\n--- {provider.upper()} ---")
        print(resultado.model_dump_json(indent=2))
    except Exception as e:
        print(f"\n--- {provider.upper()} falló: {e} ---")

ERROR:pipeline_extraccion:[openai] ❌ Falló tras reintentos: 'ascii' codec can't encode character '\xf1' in position 18: ordinal not in range(128)



--- OPENAI falló: 'ascii' codec can't encode character '\xf1' in position 18: ordinal not in range(128) ---


ERROR:pipeline_extraccion:[anthropic] ❌ Falló tras reintentos: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011Ceti1tftwZu7MVkJxfANe'}



--- ANTHROPIC falló: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011Ceti1tftwZu7MVkJxfANe'} ---

--- GEMINI ---
{
  "tecnologias": [
    "FastAPI",
    "Redis",
    "PostgreSQL"
  ],
  "nivel_de_criticidad": "alta",
  "resumen_tecnico": "La API en FastAPI sufre timeouts intermitentes en produccion debido a la saturacion del cache en Redis y el agotamiento del pool de conexiones en PostgreSQL durante picos de trafico."
}


## **Prueba de estrés (texto ambiguo)**

Pasamos un texto sin detalles técnicos claros para observar cómo se
comporta el pipeline: ¿el modelo alucina tecnologías? ¿asigna
criticidad "media" por default? Esto va documentado en el README.

In [ ]:
texto_ambiguo = "El sistema anda medio raro últimamente, no sé bien qué está pasando."

resultado = await process_text(texto_ambiguo, provider="gemini")
print(resultado.model_dump_json(indent=2))

ERROR:pipeline_extraccion:[gemini] ❌ Falló tras reintentos: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 17.283912731s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequ

GoogleRateLimitError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 17.283912731s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier'